In [ ]:
# EDA

import duckdb
import pandas as pd
import time
"""

C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_7d.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_14d.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_28d.parquet

C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_daily_impact.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_daily_status.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_windowed.parquet

C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_diff.parquet
"""
# 파일 경로 지정
parquet_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"

# 판다스 출력 제한 해제 (모든 컬럼과 행을 숨김없이 표시)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(f"[{parquet_file}]")
print("종합 EDA 및 데이터 무결성 검증을 시작합니다 ...\n")
start_time = time.time()

# DuckDB 인메모리 연결
con = duckdb.connect()

try:
    # ---------------------------------------------------------
    # 1. 데이터 규격 (행/열 개수)
    # ---------------------------------------------------------
    print("=== [1. 데이터 규격 확인] ===")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{parquet_file}')").fetchone()[0]
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    col_names = schema_df['column_name'].tolist()
    
    print(f"총 행 수(Rows): {total_rows:,} 개")
    print(f"총 열 수(Columns): {len(col_names)} 개\n")

    # # ---------------------------------------------------------
    # # 2. 상위 10개 데이터 샘플 (모든 열 표시)
    # # ---------------------------------------------------------
    # print("=== [2. 상위 10개 데이터 샘플 (생략 없음)] ===")
    # sample_df = con.execute(f"SELECT * FROM read_parquet('{parquet_file}') LIMIT 10").fetchdf()
    # print(sample_df)
    # print("\n")


    # ---------------------------------------------------------
    # 4. 모든 열에 대한 결측치(NULL) 개수 세기 (전체 열 표시)
    # ---------------------------------------------------------
    print("=== [4. 컬럼별 결측치 집계 (전체 열)] ===")
    null_count_selects = [f"COUNT(*) - COUNT(\"{col}\") AS \"{col}\"" for col in col_names]
    query_nulls = f"SELECT {', '.join(null_count_selects)} FROM read_parquet('{parquet_file}')"
    null_counts = con.execute(query_nulls).fetchdf().iloc[0]
    
    null_summary = pd.DataFrame({'Missing_Count': null_counts})
    null_summary['Missing_Ratio(%)'] = (null_summary['Missing_Count'] / total_rows) * 100
    
    # 필터링 없이 정렬만 수행하여 모든 열을 보여줌
    null_summary_sorted = null_summary.sort_values(by='Missing_Count', ascending=False)
    
    pd.set_option('display.max_rows', None)
    print(null_summary_sorted)
    pd.reset_option('display.max_rows')
    print("\n")

    # # ---------------------------------------------------------
    # # 5. 열 별 간단한 기초 통계 (최솟값, 최댓값, 평균, 표준편차)
    # # ---------------------------------------------------------
    # print("=== [5. 열 별 간단한 기초 통계 (Numeric Data)] ===")
    # summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    # stats_df = summary_df[['column_name', 'column_type', 'min', 'max', 'avg', 'std']].copy()
    
    # pd.set_option('display.max_rows', None)
    # print(stats_df)
    # pd.reset_option('display.max_rows')
    # print("\n")

    # # ---------------------------------------------------------
    # # 6. 시계열 연속성 검사 (Date Gap)
    # # ---------------------------------------------------------
    # print("=== [6. 시계열 연속성(Date Gap) 검사] ===")
    # gap_check_query = f"""
    #     WITH DateRange AS (
    #         SELECT 
    #             serial_number,
    #             MIN(CAST(date AS DATE)) as start_date,
    #             MAX(CAST(date AS DATE)) as end_date,
    #             COUNT(*) as actual_row_count,
    #             (MAX(CAST(date AS DATE)) - MIN(CAST(date AS DATE)) + 1) as expected_row_count
    #         FROM read_parquet('{parquet_file}')
    #         GROUP BY serial_number
    #     )
    #     SELECT 
    #         COUNT(*) AS serials_with_gaps,
    #         SUM(expected_row_count - actual_row_count) AS total_missing_days
    #     FROM DateRange
    #     WHERE actual_row_count != expected_row_count
    # """
    # gap_result = con.execute(gap_check_query).fetchdf().iloc[0]
    
    # if gap_result['serials_with_gaps'] == 0:
    #     print("✅ 모든 개체의 날짜가 하루도 빠짐없이 연속적입니다.")
    # else:
    #     print(f"⚠️ 날짜 공백(Gap)이 발견된 개체 수: {int(gap_result['serials_with_gaps']):,} 개")
    #     print(f"⚠️ 총 누락된 날짜(데이터 행) 수: {int(gap_result['total_missing_days']):,} 일")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    
    end_time = time.time()
    print(f"\n모든 종합 검증 완료. 총 소요 시간: {end_time - start_time:.2f}초")


<>:6: SyntaxWarning: invalid escape sequence '\W'
<>:6: SyntaxWarning: invalid escape sequence '\W'
C:\Users\joon6\AppData\Local\Temp\ipykernel_10972\931163296.py:6: SyntaxWarning: invalid escape sequence '\W'
  """


[C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_train.parquet]
종합 EDA 및 데이터 무결성 검증을 시작합니다 ...

=== [1. 데이터 규격 확인] ===
총 행 수(Rows): 299,442 개
총 열 수(Columns): 310 개

=== [4. 컬럼별 결측치 집계 (전체 열)] ===
                               Missing_Count  Missing_Ratio(%)
s183_diff                               2940          0.981826
s242_diff                               2940          0.981826
total_seeks_diff                        2940          0.981826
s190_diff                               2940          0.981826
s184_diff                               2940          0.981826
s187_diff                               2940          0.981826
s198_diff                               2940          0.981826
timeout_5s_diff                         2940          0.981826
timeout_total_diff                      2940          0.981826
seek_error_count_diff                   2940          0.981826
s189_diff                               2940          0.981826
s191_diff                          

In [3]:
import duckdb
import os

paths = [
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_diff.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_7d.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_14d.parquet',
    r'C:\Workspace\06_ML_projdect\26_1_COIN\data\fs_sample_data\fs_sample_28d.parquet'
]

# 제외할 메타데이터 컬럼 키워드
meta_keywords = ['failure', 'date', 'serial_number']

con = duckdb.connect()

print(f"{'파일명':<25} | {'전체':<5} | {'메타':<5} | {'피처(전체-메타)':<12}")
print("-" * 65)

grand_total_features = 0

for p in paths:
    if os.path.exists(p):
        # 1. 전체 컬럼 목록 가져오기
        all_cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()]
        total_cnt = len(all_cols)
        
        # 2. 메타데이터 컬럼 개수 세기 (정확히 일치하거나 키워드 포함 시)
        meta_cols = [c for c in all_cols if any(k in c.lower() for k in meta_keywords)]
        meta_cnt = len(meta_cols)
        
        # 3. 순수 피처 개수
        feature_cnt = total_cnt - meta_cnt
        grand_total_features += feature_cnt
        
        file_name = os.path.basename(p)
        print(f"{file_name:<25} | {total_cnt:<5} | {meta_cnt:<5} | {feature_cnt:<12}")
    else:
        print(f"{os.path.basename(p):<25} | 파일 없음")

print("-" * 65)
print(f"✅ 모든 파일의 순수 피처(Feature) 합계: {grand_total_features}개")

con.close()


파일명                       | 전체    | 메타    | 피처(전체-메타)   
-----------------------------------------------------------------
fs_sample_diff.parquet   | 40    | 3     | 37          
fs_sample_7d.parquet | 58    | 2     | 56          
fs_sample_14d.parquet | 76    | 2     | 74          
fs_sample_28d.parquet | 76    | 2     | 74          
-----------------------------------------------------------------
✅ 모든 파일의 순수 피처(Feature) 합계: 241개


In [18]:
import duckdb
import os

# 1. 기준이 되는 데이터 폴더 정의 (하나 밖으로 나가서 data/fs_data)
base_dir = os.path.join('..', 'data', 'fs_data')
# 2. 파일명 리스트
file_names = [
    'fs_sample_7d.parquet',
    # 'fs_sample_14d.parquet',
    # 'fs_sample_28d.parquet',

    # 'fs_sample_daily_impact.parquet',
    # 'fs_sample_daily_status.parquet',
    # 'fs_sample_windowed.parquet',

    # 'fs_sample_diff.parquet',

    # 'fs_sample_train.parquet'
]
# 3. 리스트 컴프리헨션을 사용해 전체 상대 경로 생성
paths = [os.path.join(base_dir, f) for f in file_names]


con = duckdb.connect()
all_features = set()

for p in paths:
    if os.path.exists(p):
        cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()]
        # .strip()을 추가하여 눈에 안 보이는 공백 제거
        features = [c.strip() for c in cols if not any(k in c.lower() for k in ['failure', 'date', 'serial_number'])]
        all_features.update(features)

con.close()

sorted_features = sorted(list(all_features))
print(f"고유 피처 수: {len(sorted_features)}개")
for feat in sorted_features:
    # repr()은 문자열의 실제 형태(공백 포함)를 따옴표와 함께 보여줍니다.
    print(f"{feat}") 


고유 피처 수: 56개
s184_3d_max
s184_3d_sum
s184_7d_max
s184_7d_sum
s190_7d_asfd
s190_7d_cid
s190_7d_dai
s190_7d_ewma
s190_7d_max
s190_7d_mean
s190_7d_std
s190_7d_zscore
s194_7d_asfd
s194_7d_cid
s194_7d_dai
s194_7d_ewma
s194_7d_max
s194_7d_mean
s194_7d_std
s194_7d_zscore
s241_7d_accel
s241_7d_asfd
s241_7d_dai
s241_7d_ewma
s241_7d_max
s241_7d_mean
s241_7d_std
s241_7d_sum
s241_7d_zscore
s242_7d_accel
s242_7d_asfd
s242_7d_dai
s242_7d_ewma
s242_7d_max
s242_7d_mean
s242_7d_std
s242_7d_sum
s242_7d_zscore
total_reads_7d_accel
total_reads_7d_asfd
total_reads_7d_dai
total_reads_7d_ewma
total_reads_7d_max
total_reads_7d_mean
total_reads_7d_std
total_reads_7d_sum
total_reads_7d_zscore
total_seeks_7d_accel
total_seeks_7d_asfd
total_seeks_7d_dai
total_seeks_7d_ewma
total_seeks_7d_max
total_seeks_7d_mean
total_seeks_7d_std
total_seeks_7d_sum
total_seeks_7d_zscore
